# AI Detects Abnormal Events in Sequential Data

Chapter 4 replication using UNSW-NB15, DabLog, and DeepLog. Select **Runtime → Change runtime type → T4 GPU**, then run cells in order.

In [11]:
import os
import sys

os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={sys.prefix}"
print(os.environ["XLA_FLAGS"])

--xla_gpu_cuda_data_dir=/bsuhome/lindseynielsen/miniforge3/envs/sequential-anomaly


## 1. Clone, install, and verify

In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO = Path.home() / "sequential-anomaly-detection"
os.chdir(REPO)

os.environ["PYTHONPATH"] = str(REPO / "src")

import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True,
)

2026-11-03 18:00:18.235611: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-03 18:00:18.247419: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 18:00:18.262073: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 18:00:18.266345: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 18:00:18.277183: I tensorflow/core/platform/cpu_feature_guar

Python: 3.11.17 (main, Oct  7 2026, 00:02:24) [GCC 15.3.0]
TensorFlow: 2.17.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


test_experiment_rejects_blank_name_and_negative_seed (test_config.ConfigurationTests.test_experiment_rejects_blank_name_and_negative_seed) ... ok
test_invalid_file_has_actionable_location (test_config.ConfigurationTests.test_invalid_file_has_actionable_location) ... ok
test_missing_required_settings_are_rejected (test_config.ConfigurationTests.test_missing_required_settings_are_rejected) ... ok
test_path_values_remain_portable_relative_paths (test_config.ConfigurationTests.test_path_values_remain_portable_relative_paths) ... ok
test_preprocessing_rejects_invalid_values (test_config.ConfigurationTests.test_preprocessing_rejects_invalid_values) ... ok
test_shipped_configurations_load (test_config.ConfigurationTests.test_shipped_configurations_load) ... ok
test_chapter_defaults_create_mirrored_lstm_widths (test_dablog.DablogArchitectureTests.test_chapter_defaults_create_mirrored_lstm_widths) ... ok
test_deeper_plan_follows_legacy_integer_division (test_dablog.DablogArchitectureTests.test_

CompletedProcess(args=['/bsuhome/lindseynielsen/miniforge3/envs/sequential-anomaly/bin/python', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 2. Mount Drive and validate the raw data

Place the four original headerless files UNSW-NB15_1.csv through UNSW-NB15_4.csv in MyDrive/UNSW-NB15/raw. Do not use the separate predefined training/testing CSV files.

In [4]:
RAW_DIR = Path.home() / "UNSW-NB15" / "raw"

RAW_FILES = [
    RAW_DIR / f"UNSW-NB15_{index}.csv"
    for index in range(1, 5)
]

for path in RAW_FILES:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.name, f"{path.stat().st_size / 1024**2:.1f} MB")

UNSW-NB15_1.csv 161.2 MB
UNSW-NB15_2.csv 157.6 MB
UNSW-NB15_3.csv 147.4 MB
UNSW-NB15_4.csv 93.1 MB


In [5]:
from pathlib import Path
from datetime import datetime

dataset = (
    Path.home()
    / "sequential-anomaly-detection"
    / "datasets/processed/unsw-nb15"
)

if dataset.exists():
    backup = dataset.with_name(
        "unsw-nb15-incomplete-" + datetime.now().strftime("%Y%m%d-%H%M%S")
    )
    dataset.rename(backup)
    print("Previous split moved to:", backup)

Previous split moved to: /bsuhome/lindseynielsen/sequential-anomaly-detection/datasets/processed/unsw-nb15-incomplete-20261103-180209


## 3. Reproduce the directional split

The supplied splitter creates flow_from_this.csv and flow_to_this.csv per IP. A marker prevents duplicate appends when rerunning this cell.

In [6]:
from pathlib import Path
import subprocess
import sys

# Recreate paths after a runtime restart.
REPO = Path.home() / "sequential-anomaly-detection"
RAW_DIR = Path.home() / "UNSW-NB15" / "raw"

if not REPO.is_dir():
    raise FileNotFoundError(
        "Repository not found. Run the clone/setup cell again first."
    )

RAW_FILES = [
    RAW_DIR / f"UNSW-NB15_{index}.csv"
    for index in range(1, 5)
]

missing = [
    str(path)
    for path in RAW_FILES
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "The runtime restarted and these raw files are unavailable:\n"
        + "\n".join(missing)
        + "\nUpload or download the four files again."
    )

DATASET = REPO / "datasets" / "processed" / "unsw-nb15"
MARKER = DATASET / ".split_complete"

if MARKER.exists():
    print("Reusing completed directional split.")

else:
    existing = (
        list(DATASET.glob("**/*.csv"))
        if DATASET.exists()
        else []
    )

    if existing:
        raise RuntimeError(
            "The processed directory contains CSV files but has no "
            "completion marker. Do not rerun the append-based splitter "
            "over these files. Restart the runtime or clear only the "
            "incomplete processed directory."
        )

    DATASET.mkdir(parents=True, exist_ok=True)

    command = [
        sys.executable,
        "legacy/data/unswnb15/split.py",
        *map(str, RAW_FILES),
        "--dir",
        str(DATASET),
        "--head",
        "-1",
    ]

    print("Starting directional split...")

    subprocess.run(
        command,
        cwd=REPO,
        check=True,
    )

    MARKER.write_text(
        "complete\n",
        encoding="utf-8",
    )

    print("Directional split completed.")

print("Processed data:", DATASET)
print("Completion marker:", MARKER.exists())

Starting directional split...
=====SPLIT UNSW-NB15 BEGINS=====
> File  /bsuhome/lindseynielsen/UNSW-NB15/raw/UNSW-NB15_1.csv  has been extracted successfully.
> File  /bsuhome/lindseynielsen/UNSW-NB15/raw/UNSW-NB15_2.csv  has been extracted successfully.
> File  /bsuhome/lindseynielsen/UNSW-NB15/raw/UNSW-NB15_3.csv  has been extracted successfully.
> File  /bsuhome/lindseynielsen/UNSW-NB15/raw/UNSW-NB15_4.csv  has been extracted successfully.
=====SPLIT UNSW-NB15 ENDS=====
Directional split completed.
Processed data: /bsuhome/lindseynielsen/sequential-anomaly-detection/datasets/processed/unsw-nb15
Completion marker: True


## 4. Validate and preprocess

In [7]:
PREPARED = REPO / "results/runs/shared/preprocessed"
subprocess.run([
    sys.executable, "-m", "sequential_anomaly.cli.prepare_data",
    "--config", "configs/pilot.yaml", "--output", str(PREPARED)
], check=True)
print((PREPARED / "summary.json").read_text())

{
  "experiment": "pilot",
  "partitions": {
    "testing_normal": {
      "anomalous": 0,
      "normal": 1873002,
      "windows": 1873002
    },
    "testing_victim": {
      "anomalous": 345885,
      "normal": 38,
      "windows": 345923
    },
    "training": {
      "anomalous": 0,
      "normal": 1873001,
      "windows": 1873001
    }
  },
  "preprocessing": {
    "keyset": 0,
    "packet_divisor": 100,
    "sequence_length": 10,
    "window_minutes": 15
  },
  "seed": 42,
  "seeded_libraries": {
    "numpy": true,
    "python": true,
    "tensorflow": false
  },
  "vocabulary_size": 291
}
{
  "experiment": "pilot",
  "partitions": {
    "testing_normal": {
      "anomalous": 0,
      "normal": 1873002,
      "windows": 1873002
    },
    "testing_victim": {
      "anomalous": 345885,
      "normal": 38,
      "windows": 345923
    },
    "training": {
      "anomalous": 0,
      "normal": 1873001,
      "windows": 1873001
    }
  },
  "preprocessing": {
    "keyset": 0,
    "

In [8]:
from pathlib import Path

evaluate_file = (
    REPO
    / "src"
    / "sequential_anomaly"
    / "cli"
    / "evaluate.py"
)

source = evaluate_file.read_text(encoding="utf-8")

old_import = (
    "from sequential_anomaly.training.train "
    "import read_prepared_windows\n"
)

new_import = """from sequential_anomaly.training.train import read_prepared_windows
from sequential_anomaly.data.vocabulary import EventVocabulary
from sequential_anomaly.models.deeplog import (
    DeeplogModelConfig,
    build_deeplog,
)
"""

if "from sequential_anomaly.data.vocabulary import EventVocabulary" not in source:
    if old_import not in source:
        raise RuntimeError("Could not locate the expected training import.")
    source = source.replace(old_import, new_import)

old_loading = """    model = tensorflow.keras.models.load_model(checkpoint)
"""

new_loading = """    if model_name == "dablog":
        model = tensorflow.keras.models.load_model(checkpoint)
    else:
        # Rebuild DeepLog so use_cudnn=False is retained during evaluation.
        vocabulary = EventVocabulary.load(
            prepared / "vocabulary.json"
        )
        model = build_deeplog(
            DeeplogModelConfig(
                vocabulary_size=len(vocabulary),
                sequence_length=sequence_length,
            )
        )
        model.load_weights(checkpoint)
"""

if "model.load_weights(checkpoint)" in source:
    print("DeepLog evaluation patch is already installed.")
elif old_loading not in source:
    raise RuntimeError("Could not locate the checkpoint-loading statement.")
else:
    source = source.replace(old_loading, new_loading)
    evaluate_file.write_text(source, encoding="utf-8")
    print("DeepLog evaluation patch installed.")

DeepLog evaluation patch installed.


## 5. One-epoch pilot

This verifies both complete pipelines. Pilot metrics are not final report results.

In [12]:
from itertools import islice
from pathlib import Path
import os
import shutil
import subprocess
import sys

# Create a small pilot dataset from the already prepared full dataset.
PILOT_PREPARED = REPO / "results/runs/pilot/preprocessed"
PILOT_PREPARED.mkdir(parents=True, exist_ok=True)

PILOT_TRAIN_RECORDS = 20_000
PILOT_NORMAL_RECORDS = 5_000
PILOT_VICTIM_RECORDS = 5_000


def copy_first_records(source, destination, record_count):
    source = Path(source)
    destination = Path(destination)

    with source.open("r", encoding="utf-8") as input_stream:
        with destination.open("w", encoding="utf-8") as output_stream:
            for line in islice(input_stream, record_count):
                output_stream.write(line)

    actual_count = sum(
        1
        for _ in destination.open("r", encoding="utf-8")
    )

    print(
        f"{destination.name}: "
        f"{actual_count:,} of {record_count:,} requested records"
    )


# Preserve the vocabulary fitted on the full normal training partition.
shutil.copy2(
    PREPARED / "vocabulary.json",
    PILOT_PREPARED / "vocabulary.json",
)

copy_first_records(
    PREPARED / "train.jsonl",
    PILOT_PREPARED / "train.jsonl",
    PILOT_TRAIN_RECORDS,
)

copy_first_records(
    PREPARED / "test_normal.jsonl",
    PILOT_PREPARED / "test_normal.jsonl",
    PILOT_NORMAL_RECORDS,
)

copy_first_records(
    PREPARED / "test_victim.jsonl",
    PILOT_PREPARED / "test_victim.jsonl",
    PILOT_VICTIM_RECORDS,
)

# Ensure subprocesses can import the src-layout package.
command_environment = os.environ.copy()
command_environment["PYTHONPATH"] = str(REPO / "src")

for model_name in ("dablog", "deeplog"):
    print(f"\n{'=' * 70}")
    print(f"Running pilot for {model_name.upper()}")
    print(f"{'=' * 70}\n")

    checkpoint_dir = (
        REPO
        / "artifacts"
        / "checkpoints"
        / "pilot"
        / model_name
    )

    evaluation_dir = (
        REPO
        / "results"
        / "runs"
        / "pilot"
        / model_name
    )

    training_command = [
        sys.executable,
        "-m",
        "sequential_anomaly.cli.train",
        "--config",
        "configs/pilot.yaml",
        "--model",
        model_name,
        "--prepared",
        str(PILOT_PREPARED),
        "--output",
        str(checkpoint_dir),
        "--epochs",
        "1",
        "--batch-size",
        "64",
        "--patience",
        "1",
    ]

    training_result = subprocess.run(
        training_command,
        text=True,
        capture_output=True,
        env=command_environment,
    )

    print(training_result.stdout)
    print(training_result.stderr)

    if training_result.returncode != 0:
        raise RuntimeError(
            f"{model_name} pilot training failed with "
            f"exit code {training_result.returncode}"
        )

    evaluation_command = [
        sys.executable,
        "-m",
        "sequential_anomaly.cli.evaluate",
        "--config",
        "configs/pilot.yaml",
        "--model",
        model_name,
        "--prepared",
        str(PILOT_PREPARED),
        "--checkpoint",
        str(checkpoint_dir / "best.keras"),
        "--output",
        str(evaluation_dir),
        "--batch-size",
        "64",
    ]

    evaluation_result = subprocess.run(
        evaluation_command,
        text=True,
        capture_output=True,
        env=command_environment,
    )

    print(evaluation_result.stdout)
    print(evaluation_result.stderr)

    if evaluation_result.returncode != 0:
        raise RuntimeError(
            f"{model_name} pilot evaluation failed with "
            f"exit code {evaluation_result.returncode}"
        )

    print(f"{model_name.upper()} pilot completed successfully.")

train.jsonl: 20,000 of 20,000 requested records
test_normal.jsonl: 5,000 of 5,000 requested records
test_victim.jsonl: 5,000 of 5,000 requested records

Running pilot for DABLOG


  1/313 ━━━━━━━━━━━━━━━━━━━━ 51:08 10s/step - accuracy: 0.0766 - loss: 5.673
  4/313 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.1915 - loss: 5.6705 
  7/313 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.2246 - loss: 5.667
 10/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2419 - loss: 5.663
 13/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2536 - loss: 5.658
 16/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2619 - loss: 5.650
 19/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2686 - loss: 5.633
 22/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2737 - loss: 5.598
 25/313 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.2777 - loss: 5.550
 28/313 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.2806 - loss: 5.492
 31/313 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.2828 - los

In [13]:
import os
import subprocess
import sys

command_environment = os.environ.copy()
command_environment["PYTHONPATH"] = str(REPO / "src")

checkpoint_dir = (
    REPO
    / "artifacts"
    / "checkpoints"
    / "pilot"
    / "deeplog"
)

evaluation_dir = (
    REPO
    / "results"
    / "runs"
    / "pilot"
    / "deeplog"
)

checkpoint = checkpoint_dir / "best.keras"

if not checkpoint.is_file():
    raise FileNotFoundError(
        f"DeepLog checkpoint was not found: {checkpoint}"
    )

evaluation_command = [
    sys.executable,
    "-m",
    "sequential_anomaly.cli.evaluate",
    "--config",
    "configs/pilot.yaml",
    "--model",
    "deeplog",
    "--prepared",
    str(PILOT_PREPARED),
    "--checkpoint",
    str(checkpoint),
    "--output",
    str(evaluation_dir),
    "--batch-size",
    "64",
]

subprocess.run(
    evaluation_command,
    check=True,
    env=command_environment,
)

print("DeepLog pilot evaluation completed successfully.")
print("Results:", evaluation_dir)

2026-11-03 20:05:36.286183: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-03 20:05:36.296775: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-03 20:05:36.308932: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-03 20:05:36.312372: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-11-03 20:05:36.321629: I tensorflow/core/platform/cpu_feature_guar

{
  "groups": 147,
  "anomalous_groups": 113,
  "normal_groups": 34,
  "at_threshold": {
    "threshold": 0.05,
    "true_positive": 109,
    "false_positive": 32,
    "true_negative": 2,
    "false_negative": 4,
    "precision": 0.7730496453900709,
    "recall": 0.9646017699115044,
    "specificity": 0.058823529411764705,
    "accuracy": 0.7551020408163265,
    "f1": 0.8582677165354331
  },
  "auroc": 0.8720718375845913,
  "auprc": 0.9359376891852471,
  "best_f1": 0.919831223628692,
  "best_f1_threshold": 0.10996563573883161
}
DeepLog pilot evaluation completed successfully.
Results: /bsuhome/lindseynielsen/sequential-anomaly-detection/results/runs/pilot/deeplog


## 6. Full replication

Review the pilot first, then set RUN_FULL_EXPERIMENT to True. The full run preserves the supplied 32-epoch UNSW driver setting and model-specific batch sizes.

In [ ]:
RUN_FULL_EXPERIMENT = False
if RUN_FULL_EXPERIMENT:
    for model_name in ("dablog", "deeplog"):
        config = f"configs/{model_name}.yaml"
        checkpoint_dir = REPO / "artifacts/checkpoints" / model_name
        subprocess.run([
            sys.executable, "-m", "sequential_anomaly.cli.train",
            "--config", config, "--prepared", str(PREPARED),
            "--output", str(checkpoint_dir), "--epochs", "32"
        ], check=True)
        subprocess.run([
            sys.executable, "-m", "sequential_anomaly.cli.evaluate",
            "--config", config, "--prepared", str(PREPARED),
            "--checkpoint", str(checkpoint_dir / "best.keras")
        ], check=True)
else:
    print("Full experiment disabled until the pilot is reviewed.")

## 7. Compare metrics and charts

In [ ]:
from IPython.display import Image, display
for model_name in ("dablog", "deeplog"):
    evaluation = (REPO / f"results/runs/{model_name}/evaluation" if RUN_FULL_EXPERIMENT
                  else REPO / f"results/runs/pilot/{model_name}")
    metrics = json.loads((evaluation / "metrics.json").read_text())
    print(model_name.upper(), {k: metrics[k] for k in
          ("auroc", "auprc", "best_f1", "best_f1_threshold")})
    display(Image(filename=str(evaluation / "roc_curve.png")))
    display(Image(filename=str(evaluation / "precision_recall_curve.png")))